# Notebook 04: Forecast Evaluation & Error Analysis

**Dataset**: FreshRetailNet-50K (Fresh grocery retail demand data)

---

## Motivation

Building a forecast model is only half the job. The other half -- and arguably the more important half -- is **understanding where and why the model fails**. A single aggregate metric like MAE hides enormous variation across different product segments, demand levels, and time periods.

In this notebook we will:

1. Train a multi-model LightGBM ensemble and generate predictions on the held-out eval period  
2. Compute comprehensive error metrics overall  
3. **Deep-dive into SMAPE vs WMAE** -- why SMAPE is misleading for heterogeneous retail demand  
4. Segment errors by stockout rate, day of week, and product category  
5. Evaluate **prediction interval calibration** -- are our uncertainty bands trustworthy?  
6. Visualize diagnostic time series for individual store-products  

### Key Teaching Points

| Concept | Why It Matters |
|---------|---------------|
| SMAPE vs WMAE | SMAPE treats all observations equally; WMAE weights by economic importance |
| PI Calibration | Under-coverage means inventory decisions based on PIs will stockout more than expected |
| Segment Analysis | Aggregate metrics hide where improvement effort should be focused |

---

## Section 1: Setup & Generate Eval Predictions

We load the data, sample 3000 store-products stratified by stockout rate, engineer 90+ features, and train five LightGBM models (MAE, Huber, Q10, Q50, Q90). Then we generate predictions on the 7-day eval period.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings, gc

import os
NB_OUT = os.path.join('..', 'notebook_output')
os.makedirs(NB_OUT, exist_ok=True)

warnings.filterwarnings('ignore')
np.random.seed(42)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("Libraries loaded.")

### 1.1 Load Data & Stratified Sampling (3000 Store-Products)

In [ ]:
TRAIN_PATH = '../../data/freshretailnet/raw/data/train.parquet'
EVAL_PATH  = '../../data/freshretailnet/raw/data/eval.parquet'
N_SP = 3000

cols = [
    'city_id', 'store_id', 'management_group_id', 'first_category_id',
    'second_category_id', 'third_category_id', 'product_id', 'dt',
    'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag',
    'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'
]

# --- Load training data ---
train_raw = pd.read_parquet(TRAIN_PATH, columns=cols)
train_raw['sp'] = train_raw['store_id'] * 10000 + train_raw['product_id']

# --- Stratified sample by stockout rate ---
so_rate = train_raw.groupby('sp')['stock_hour6_22_cnt'].apply(lambda x: (x > 0).mean())
so_rate = so_rate.reset_index()
so_rate.columns = ['sp', 'stockout_rate']
so_rate['bin'] = pd.qcut(so_rate['stockout_rate'], q=5, labels=False, duplicates='drop')

sampled_sps = (
    so_rate.groupby('bin')
    .apply(lambda x: x.sample(min(len(x), N_SP // 5), random_state=42))
    .reset_index(drop=True)['sp'].values
)
if len(sampled_sps) < N_SP:
    remaining = np.setdiff1d(so_rate['sp'].values, sampled_sps)
    extra = np.random.choice(remaining, N_SP - len(sampled_sps), replace=False)
    sampled_sps = np.concatenate([sampled_sps, extra])
sampled_sps = set(sampled_sps[:N_SP])

train = train_raw[train_raw['sp'].isin(sampled_sps)].copy()
train['dt'] = pd.to_datetime(train['dt'])

# --- Load eval data ---
eval_raw = pd.read_parquet(EVAL_PATH, columns=cols)
eval_raw['sp'] = eval_raw['store_id'] * 10000 + eval_raw['product_id']
ev = eval_raw[eval_raw['sp'].isin(sampled_sps)].copy()
ev['dt'] = pd.to_datetime(ev['dt'])

# --- Downcast dtypes for memory efficiency ---
for c in train.select_dtypes('int64').columns:
    if c != 'sp':
        train[c] = train[c].astype('int32')
        ev[c] = ev[c].astype('int32')
for c in train.select_dtypes('float64').columns:
    train[c] = train[c].astype('float32')
    ev[c] = ev[c].astype('float32')

print(f"Train shape: {train.shape}")
print(f"Eval shape:  {ev.shape}")
print(f"Train dates: {train['dt'].min().date()} to {train['dt'].max().date()}")
print(f"Eval dates:  {ev['dt'].min().date()} to {ev['dt'].max().date()}")
print(f"Unique store-products: {train['sp'].nunique()}")
print(f"Stockout rate (train): {(train['stock_hour6_22_cnt'] > 0).mean()*100:.1f}%")

del train_raw, eval_raw, so_rate
gc.collect()

### 1.2 Censored Demand Recovery (Proportional Method)

When stockouts occur, observed sales understate true demand. We use a proportional recovery method:
- **Partial stockout** (some hours OOS): scale up by operating-hours / available-hours
- **Full stockout** (all hours OOS): impute from recent non-stockout days

In [ ]:
def add_demand_recovery(df):
    """Add censored demand recovery using the proportional method."""
    OP = 16  # operating hours (6am-10pm)
    df = df.sort_values(['sp', 'dt']).reset_index(drop=True)
    df['cens'] = (df['stock_hour6_22_cnt'] > 0).astype('int8')
    df['so_frac'] = (df['stock_hour6_22_cnt'] / OP).astype('float32')
    df['dem_rec'] = df['sale_amount'].values.copy().astype(np.float64)

    # Partial stockout: proportional scaling
    partial = (df['stock_hour6_22_cnt'] > 0) & (df['stock_hour6_22_cnt'] < OP)
    avail = (OP - df['stock_hour6_22_cnt']).clip(lower=1)
    df.loc[partial, 'dem_rec'] = df.loc[partial, 'sale_amount'] * (OP / avail[partial])

    # Full stockout: use rolling mean of non-stockout days
    full = df['stock_hour6_22_cnt'] >= OP
    no_so = (df['cens'] == 0).astype(float)
    df['_s'] = df['sale_amount'] * no_so
    rs = df.groupby('sp')['_s'].transform(lambda x: x.rolling(14, min_periods=1).sum())
    rc = df.groupby('sp').apply(lambda g: no_so.loc[g.index].rolling(14, min_periods=1).sum())
    rc = rc.reset_index(level=0, drop=True).sort_index()
    avg_nonstockout = rs / rc.clip(lower=1)
    df.loc[full, 'dem_rec'] = avg_nonstockout[full]

    # Variability correction for censored observations
    sp_std = df.groupby('sp')['sale_amount'].transform('std').fillna(0)
    correction = sp_std * df['so_frac'] * 0.3
    df.loc[df['cens'] == 1, 'dem_rec'] += correction[df['cens'] == 1]
    df['dem_rec'] = df['dem_rec'].clip(lower=0).astype('float32')
    df.drop('_s', axis=1, inplace=True)

    return df

train = add_demand_recovery(train)
print(f"Censored observations: {train['cens'].sum():,} ({train['cens'].mean()*100:.1f}%)")
print(f"Sale mean: {train['sale_amount'].mean():.4f}, Recovered demand mean: {train['dem_rec'].mean():.4f}")
gc.collect()

### 1.3 Full Feature Engineering

We build 90+ features across these categories:
- **Temporal**: day-of-week, month, cyclical encodings
- **Lags**: 1,2,3,5,7,14,21,28-day lags (observed + recovered)
- **Rolling**: mean, std, max, min, median, quantiles over 3/7/14/28-day windows
- **EWMA**: exponentially-weighted moving averages (span 7, 14)
- **Stockout**: recent stockout rate, stockout hours
- **Cross features**: discount x holiday, temperature x precipitation, etc.
- **Global stats**: mean/std per store-product, product, store, city, category
- **Hierarchy features**: category-level aggregates
- **Clustering**: KMeans (k=25) on store-product behavioral features

In [ ]:
def make_features(df):
    """Build the full feature set (90+ features)."""
    df = df.sort_values(['sp', 'dt']).reset_index(drop=True)
    g = df.groupby('sp')

    # === TEMPORAL ===
    df['dow']   = df['dt'].dt.dayofweek.astype('int8')
    df['dom']   = df['dt'].dt.day.astype('int8')
    df['woy']   = df['dt'].dt.isocalendar().week.astype('int8')
    df['month'] = df['dt'].dt.month.astype('int8')
    df['wknd']  = (df['dow'] >= 5).astype('int8')
    df['doy']   = df['dt'].dt.dayofyear.astype('int16')
    for cyc, period in [('dow', 7), ('dom', 31), ('woy', 52)]:
        df[f'{cyc}_sin'] = np.sin(2 * np.pi * df[cyc] / period).astype('float32')
        df[f'{cyc}_cos'] = np.cos(2 * np.pi * df[cyc] / period).astype('float32')

    # === LAGS ===
    targets = {'s': 'sale_amount', 'r': 'dem_rec'}
    for pfx, col in targets.items():
        for lag in [1, 2, 3, 5, 7, 14, 21, 28]:
            df[f'{pfx}_l{lag}'] = g[col].shift(lag).astype('float32')

    # Diff / momentum
    df['s_d1'] = (df['s_l1'] - df['s_l2']).astype('float32')
    df['s_d7'] = (df['s_l1'] - df['s_l7']).astype('float32')
    df['r_d1'] = (df['r_l1'] - g['dem_rec'].shift(2)).astype('float32')

    # === ROLLING STATS ===
    shifted_s = g['sale_amount'].shift(1)
    shifted_r = g['dem_rec'].shift(1)

    for pfx, shifted in [('s', shifted_s), ('r', shifted_r)]:
        for w in [3, 7, 14, 28]:
            r = shifted.groupby(df['sp']).rolling(w, min_periods=1)
            df[f'{pfx}_m{w}']  = r.mean().reset_index(level=0, drop=True).astype('float32')
            df[f'{pfx}_sd{w}'] = r.std().reset_index(level=0, drop=True).astype('float32')
            if w in [7, 28]:
                df[f'{pfx}_mx{w}'] = r.max().reset_index(level=0, drop=True).astype('float32')
                df[f'{pfx}_mn{w}'] = r.min().reset_index(level=0, drop=True).astype('float32')
                df[f'{pfx}_md{w}'] = r.median().reset_index(level=0, drop=True).astype('float32')
            gc.collect()

    # Quantile features
    for w in [14, 28]:
        r = shifted_s.groupby(df['sp']).rolling(w, min_periods=1)
        df[f's_q25_{w}'] = r.quantile(0.25).reset_index(level=0, drop=True).astype('float32')
        df[f's_q75_{w}'] = r.quantile(0.75).reset_index(level=0, drop=True).astype('float32')

    # === EWMA ===
    for pfx, shifted in [('s', shifted_s), ('r', shifted_r)]:
        for s in [7, 14]:
            df[f'{pfx}_ew{s}'] = shifted.groupby(df['sp']).transform(
                lambda x: x.ewm(span=s, min_periods=1).mean()
            ).astype('float32')

    del shifted_s, shifted_r
    gc.collect()

    # === STOCKOUT FEATURES ===
    cs = g['cens'].shift(1)
    df['so1'] = cs.astype('float32')
    df['so7'] = g['cens'].shift(7).astype('float32')
    for w in [7, 14]:
        df[f'sor{w}'] = cs.groupby(df['sp']).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True).astype('float32')
    hs = g['stock_hour6_22_cnt'].shift(1)
    for w in [7, 14]:
        df[f'soh{w}'] = hs.groupby(df['sp']).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True).astype('float32')
    del cs, hs
    gc.collect()

    # === VARIABILITY & TREND ===
    df['cv7']     = (df['s_sd7']  / df['s_m7'].clip(lower=0.01)).astype('float32')
    df['cv28']    = (df['s_sd28'] / df['s_m28'].clip(lower=0.01)).astype('float32')
    df['tr_7_28'] = (df['s_m7']  - df['s_m28']).astype('float32')
    df['tr_3_14'] = (df['s_m3']  - df['s_m14']).astype('float32')

    # Ratio features
    df['l1_m7']        = (df['s_l1'] / df['s_m7'].clip(lower=0.01)).astype('float32')
    df['m7_m28']       = (df['s_m7'] / df['s_m28'].clip(lower=0.01)).astype('float32')
    df['rec_obs_ratio'] = (df['r_m7'] / df['s_m7'].clip(lower=0.01)).astype('float32')

    # DOW profile
    df['dow_prof']   = df.groupby(['sp', 'dow'])['sale_amount'].transform('mean').astype('float32')
    df['dow_prof_r'] = df.groupby(['sp', 'dow'])['dem_rec'].transform('mean').astype('float32')

    # === CROSS FEATURES ===
    df['d_h']      = (df['discount'] * df['holiday_flag']).astype('float32')
    df['d_a']      = (df['discount'] * df['activity_flag']).astype('float32')
    df['t_p']      = (df['avg_temperature'] * df['precpt']).astype('float32')
    df['w_h']      = (df['wknd'] * df['holiday_flag']).astype('int8')
    df['h_a']      = (df['holiday_flag'] * df['activity_flag']).astype('int8')
    df['temp_dev'] = (df['avg_temperature'] - df.groupby('sp')['avg_temperature'].transform('mean')).astype('float32')
    df['hum_dev']  = (df['avg_humidity']    - df.groupby('sp')['avg_humidity'].transform('mean')).astype('float32')

    # === GLOBAL STATS ===
    for grp, pfx in [('sp', 'sp'), ('product_id', 'pd'), ('store_id', 'st'), ('city_id', 'ct')]:
        df[f'{pfx}_m'] = df.groupby(grp)['sale_amount'].transform('mean').astype('float32')
        df[f'{pfx}_s'] = df.groupby(grp)['sale_amount'].transform('std').fillna(0).astype('float32')
    df['cat1_m'] = df.groupby('first_category_id')['sale_amount'].transform('mean').astype('float32')
    df['cat2_m'] = df.groupby('second_category_id')['sale_amount'].transform('mean').astype('float32')

    # === HIERARCHY FEATURES ===
    df['store_cat1_m'] = df.groupby(['store_id', 'first_category_id'])['sale_amount'].transform('mean').astype('float32')
    df['city_cat1_m']  = df.groupby(['city_id', 'first_category_id'])['sale_amount'].transform('mean').astype('float32')
    df['store_cat2_m'] = df.groupby(['store_id', 'second_category_id'])['sale_amount'].transform('mean').astype('float32')

    # === CLUSTERING (KMeans k=25) ===
    # Build per-SP behavioral profile for clustering
    sp_profile = df.groupby('sp').agg(
        mean_sale=('sale_amount', 'mean'),
        std_sale=('sale_amount', 'std'),
        mean_rec=('dem_rec', 'mean'),
        cens_rate=('cens', 'mean'),
        mean_disc=('discount', 'mean'),
    ).fillna(0)
    kmeans = KMeans(n_clusters=25, random_state=42, n_init=10, max_iter=100)
    sp_profile['cluster'] = kmeans.fit_predict(sp_profile.values)
    cluster_map = sp_profile['cluster'].to_dict()
    df['cluster'] = df['sp'].map(cluster_map).astype('int8')
    df['cluster_m'] = df.groupby('cluster')['sale_amount'].transform('mean').astype('float32')

    df = df.fillna(0)
    gc.collect()
    return df


def get_feature_cols(df):
    """Return feature columns (everything except identifiers and targets)."""
    exclude = {'dt', 'sale_amount', 'dem_rec', 'sp', 'cens', 'so_frac'}
    return [c for c in df.columns if c not in exclude]


print("Building features on training data...")
train = make_features(train)
feature_cols = get_feature_cols(train)
print(f"Feature count: {len(feature_cols)}")
print(f"Training data shape: {train.shape}")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1e6:.0f} MB")

### 1.4 Train LightGBM Models

We train five models with the same base hyperparameters:
- **MAE** and **Huber**: point forecasts with censored-observation downweighting (weight=0.5)
- **Q10, Q50, Q90**: quantile models for prediction intervals

The last 7 days of training are held out as validation for early stopping.

In [ ]:
LGB_BASE = {
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'min_child_samples': 30,
    'lambda_l1': 0.1,
    'lambda_l2': 1.0,
    'max_depth': -1,
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42,
}

TARGET = 'sale_amount'

# --- Train / validation split (last 7 days of training for early stopping) ---
warmup_date = train['dt'].min() + pd.Timedelta(days=28)
val_cutoff  = train['dt'].max() - pd.Timedelta(days=6)

tr_mask = (train['dt'] >= warmup_date) & (train['dt'] < val_cutoff)
vl_mask = train['dt'] >= val_cutoff

X_train, y_train = train.loc[tr_mask, feature_cols].values, train.loc[tr_mask, TARGET].values
X_val,   y_val   = train.loc[vl_mask, feature_cols].values, train.loc[vl_mask, TARGET].values

# Censored observation weights (downweight censored rows for MAE/Huber)
w_train = np.ones(len(y_train), dtype='float32')
cens_mask = train.loc[tr_mask, 'cens'].values == 1
w_train[cens_mask] = 0.5

print(f"Training rows:   {len(y_train):,} (after 28-day warmup)")
print(f"Validation rows: {len(y_val):,} (last 7 days)")
print(f"Features:        {len(feature_cols)}")
print(f"Censored in train: {cens_mask.sum():,} ({cens_mask.mean()*100:.1f}%)")

In [ ]:
models = {}

# --- Weighted datasets for MAE / Huber ---
dtrain_w = lgb.Dataset(X_train, y_train, weight=w_train, feature_name=feature_cols)
dval_w   = lgb.Dataset(X_val, y_val, feature_name=feature_cols, reference=dtrain_w)

# --- Unweighted datasets for quantile models ---
dtrain_u = lgb.Dataset(X_train, y_train, feature_name=feature_cols)
dval_u   = lgb.Dataset(X_val, y_val, feature_name=feature_cols, reference=dtrain_u)

callbacks = [lgb.early_stopping(50), lgb.log_evaluation(500)]

# 1. MAE model
print("=" * 60)
print("[1/5] Training MAE model...")
params_mae = {**LGB_BASE, 'objective': 'regression_l1', 'metric': 'mae'}
models['mae'] = lgb.train(params_mae, dtrain_w, 800,
                          valid_sets=[dtrain_w, dval_w], valid_names=['train', 'val'],
                          callbacks=callbacks)
print(f"   Best iteration: {models['mae'].best_iteration}")

# 2. Huber model
print("\n" + "=" * 60)
print("[2/5] Training Huber model...")
params_hub = {**LGB_BASE, 'objective': 'huber', 'metric': 'mae', 'huber_delta': 0.5}
models['huber'] = lgb.train(params_hub, dtrain_w, 800,
                            valid_sets=[dtrain_w, dval_w], valid_names=['train', 'val'],
                            callbacks=callbacks)
print(f"   Best iteration: {models['huber'].best_iteration}")

# 3-5. Quantile models
for q, name in [(0.1, 'q10'), (0.5, 'q50'), (0.9, 'q90')]:
    idx = [0.1, 0.5, 0.9].index(q)
    print(f"\n{'=' * 60}")
    print(f"[{3+idx}/5] Training {name.upper()} model (alpha={q})...")
    params_q = {**LGB_BASE, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'}
    models[name] = lgb.train(params_q, dtrain_u, 800,
                             valid_sets=[dtrain_u, dval_u], valid_names=['train', 'val'],
                             callbacks=callbacks)
    print(f"   Best iteration: {models[name].best_iteration}")

print("\nAll 5 models trained successfully.")

del dtrain_w, dval_w, dtrain_u, dval_u
gc.collect()

### 1.5 Build Eval Features & Generate Predictions

To generate features for the eval period, we concatenate the last 28 days of training with the 7 eval days, run feature engineering, and extract only the eval-date rows.

In [ ]:
# Prepare eval data
ev['cens']    = (ev['stock_hour6_22_cnt'] > 0).astype('int8')
ev['so_frac'] = (ev['stock_hour6_22_cnt'] / 16).astype('float32')
ev['dem_rec'] = ev['sale_amount'].astype('float32')

# Concatenate last 28 days of training + eval for feature engineering
hist_cutoff = train['dt'].max() - pd.Timedelta(days=27)
hist = train[train['dt'] >= hist_cutoff].copy()
combined = pd.concat([hist, ev], ignore_index=True)
del hist
gc.collect()

print("Building eval features (this may take a minute)...")
combined = make_features(combined)

# Extract eval-period rows
eval_dates = sorted(ev['dt'].unique())
eval_mask = combined['dt'].isin(eval_dates)
eval_df = combined[eval_mask].copy()

# Ensure all feature columns exist
for c in feature_cols:
    if c not in eval_df.columns:
        eval_df[c] = 0

X_eval = eval_df[feature_cols].fillna(0).values
y_eval = eval_df['sale_amount'].values
sp_eval = eval_df['sp'].values
dt_eval = eval_df['dt'].values

# Keep category and store info for segmentation later
cat1_eval = eval_df['first_category_id'].values
dow_eval  = eval_df['dow'].values if 'dow' in eval_df.columns else pd.to_datetime(eval_df['dt']).dt.dayofweek.values

print(f"Eval prediction rows: {len(y_eval):,}")

# --- Generate predictions ---
preds = {}
for name, mdl in models.items():
    preds[name] = np.clip(mdl.predict(X_eval), 0, None)

# Ensemble = average of MAE + Huber + Q50
preds['ensemble'] = np.clip((preds['mae'] + preds['huber'] + preds['q50']) / 3, 0, None)

print(f"\nPredictions generated for {len(preds)} models + ensemble.")

# Store per-SP average demand from training (needed for segmentation)
sp_avg_demand = train.groupby('sp')['sale_amount'].mean().to_dict()
sp_stockout_rate = train.groupby('sp')['cens'].mean().to_dict()

del combined, eval_df
gc.collect()

---

## Section 2: Overall Metrics

We compute a comprehensive set of error metrics for each model. Understanding the difference between these metrics is fundamental to good forecasting practice:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MAE | mean(\|y-p\|) | Average absolute error in original units |
| RMSE | sqrt(mean((y-p)^2)) | Penalizes large errors more than MAE |
| SMAPE | mean(2\|y-p\|/(\|y\|+\|p\|)) x 100 | Symmetric percentage error (0-200%) |
| MAPE | mean(\|y-p\|/y) on y>0 | Percentage error (undefined at zero) |
| WMAE | sum(\|y-p\|*y)/sum(y) | Weights errors by actual demand |
| Bias | mean(p-y) | Systematic over/under-prediction |
| Correlation | corr(y,p) | Linear agreement |

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute a comprehensive set of forecast error metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    nz = y_true > 0
    mape = np.mean(np.abs(y_true[nz] - y_pred[nz]) / y_true[nz]) * 100 if nz.any() else np.nan
    corr = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan
    wmae = np.sum(np.abs(y_true - y_pred) * y_true) / np.sum(y_true) if np.sum(y_true) > 0 else np.nan
    bias = np.mean(y_pred - y_true)
    return {'MAE': mae, 'RMSE': rmse, 'SMAPE': smape, 'MAPE': mape,
            'Corr': corr, 'WMAE': wmae, 'Bias': bias}


# --- Compute metrics for each model ---
model_names = ['mae', 'huber', 'q50', 'ensemble']
metrics_dict = {}
for name in model_names:
    metrics_dict[name] = compute_metrics(y_eval, preds[name])

metrics_df = pd.DataFrame(metrics_dict).T
metrics_df.index.name = 'Model'

print("=" * 80)
print("OVERALL FORECAST METRICS (EVAL PERIOD)")
print("=" * 80)
display(metrics_df.round(4))
metrics_df.to_csv(os.path.join(NB_OUT, 'nb04_overall_metrics.csv'))
print(f'Saved nb04_overall_metrics.csv to {NB_OUT}')

In [ ]:
# --- Prediction Interval Coverage ---
pi_coverage = np.mean((y_eval >= preds['q10']) & (y_eval <= preds['q90']))
print(f"80% Prediction Interval Coverage: {pi_coverage*100:.1f}%")
print(f"Expected: 80.0%")
print(f"Gap: {(pi_coverage - 0.80)*100:+.1f} percentage points")

### 2.1 Visualizations: Predicted vs Actual, Residuals, Prediction Intervals

In [ ]:
p_ens = preds['ensemble']
residuals = p_ens - y_eval

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Predicted vs Actual Scatter ---
ax = axes[0]
n_subsample = min(8000, len(y_eval))
idx_samp = np.random.choice(len(y_eval), n_subsample, replace=False)
ax.scatter(y_eval[idx_samp], p_ens[idx_samp], alpha=0.1, s=4, c='steelblue', edgecolors='none')
mx = max(np.percentile(y_eval, 99), np.percentile(p_ens, 99))
ax.plot([0, mx], [0, mx], 'r--', lw=1.5, label='Perfect Forecast')
ax.set_xlabel('Actual Demand')
ax.set_ylabel('Predicted Demand')
corr_val = np.corrcoef(y_eval, p_ens)[0, 1]
ax.set_title(f'Predicted vs Actual (Corr = {corr_val:.3f})')
ax.set_xlim(0, mx)
ax.set_ylim(0, mx)
ax.legend(fontsize=9)

# --- 2. Residual Distribution ---
ax = axes[1]
ax.hist(residuals, bins=100, density=True, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(0, color='red', ls='--', lw=1.5, label='Zero')
ax.axvline(residuals.mean(), color='orange', ls=':', lw=2,
           label=f'Mean = {residuals.mean():.4f}')
ax.set_xlabel('Residual (Predicted - Actual)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend(fontsize=9)

# --- 3. Prediction Intervals ---
ax = axes[2]
n_pi = 300
pi_idx = np.random.choice(len(y_eval), min(n_pi, len(y_eval)), replace=False)
sort_order = np.argsort(y_eval[pi_idx])
pi_idx_sorted = pi_idx[sort_order]
xs = np.arange(len(pi_idx_sorted))
ax.fill_between(xs, preds['q10'][pi_idx_sorted], preds['q90'][pi_idx_sorted],
                alpha=0.3, color='steelblue', label='80% PI (Q10-Q90)')
ax.plot(xs, y_eval[pi_idx_sorted], 'ko', ms=2, label='Actual')
ax.plot(xs, p_ens[pi_idx_sorted], 'r-', lw=0.7, alpha=0.7, label='Ensemble Forecast')
ax.set_xlabel('Sample (sorted by actual demand)')
ax.set_ylabel('Demand')
ax.set_title(f'Prediction Intervals (coverage = {pi_coverage*100:.1f}%)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_predicted_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## Section 3: SMAPE Analysis by Volume Segment (Educational Core)

### Why SMAPE is Misleading for Retail Forecasting

SMAPE (Symmetric Mean Absolute Percentage Error) treats all observations equally. An error of 0.05 kg on a product that sells 0.1 kg/day contributes the **same SMAPE** as an error of 0.5 kg on a product that sells 10 kg/day.

But economically, the 0.5 kg error on the high-volume product matters **far more** -- it represents more lost revenue, more waste, and more inventory cost.

This section demonstrates this effect and introduces **WMAE** (Weighted Mean Absolute Error) as a more appropriate metric for retail demand forecasting.

In [ ]:
# --- Compute average demand per SP from training data ---
eval_sp_avg = np.array([sp_avg_demand.get(sp, 0) for sp in sp_eval])

# --- Define volume segments ---
def assign_volume_segment(avg_demand):
    if avg_demand < 0.5:
        return 'Very Low (< 0.5)'
    elif avg_demand < 1.0:
        return 'Low (0.5 - 1.0)'
    elif avg_demand < 2.0:
        return 'Medium (1.0 - 2.0)'
    elif avg_demand < 5.0:
        return 'High (2.0 - 5.0)'
    else:
        return 'Very High (5.0+)'

segment_labels = np.array([assign_volume_segment(d) for d in eval_sp_avg])
segment_order = ['Very Low (< 0.5)', 'Low (0.5 - 1.0)', 'Medium (1.0 - 2.0)',
                 'High (2.0 - 5.0)', 'Very High (5.0+)']

# --- Per-segment metrics ---
seg_results = []
total_abs_error = np.sum(np.abs(y_eval - p_ens))

for seg in segment_order:
    mask = segment_labels == seg
    if mask.sum() == 0:
        continue
    y_seg = y_eval[mask]
    p_seg = p_ens[mask]
    m = compute_metrics(y_seg, p_seg)
    seg_abs_error = np.sum(np.abs(y_seg - p_seg))
    error_share = seg_abs_error / total_abs_error * 100
    seg_results.append({
        'Segment': seg,
        'Count': int(mask.sum()),
        'MAE': m['MAE'],
        'RMSE': m['RMSE'],
        'SMAPE': m['SMAPE'],
        'Bias': m['Bias'],
        'WMAE': m['WMAE'],
        'Error Share %': error_share,
    })

seg_df = pd.DataFrame(seg_results)
print("=" * 90)
print("METRICS BY VOLUME SEGMENT")
print("=" * 90)
display(seg_df.round(4))
seg_df.to_csv(os.path.join(NB_OUT, 'nb04_metrics_by_volume_segment.csv'), index=False)
print(f'Saved nb04_metrics_by_volume_segment.csv to {NB_OUT}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- 1. Horizontal Bar: SMAPE by Segment ---
ax = axes[0]
colors = sns.color_palette('YlOrRd', n_colors=len(seg_df))
bars = ax.barh(seg_df['Segment'], seg_df['SMAPE'], color=colors, edgecolor='gray', lw=0.5)
ax.set_xlabel('SMAPE (%)')
ax.set_title('SMAPE by Volume Segment\n(Low-volume items have highest SMAPE)')
for bar, val in zip(bars, seg_df['SMAPE']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.invert_yaxis()

# --- 2. Pie Chart / Stacked Bar: Error Share ---
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    seg_df['Error Share %'], labels=seg_df['Segment'],
    autopct='%1.1f%%', colors=sns.color_palette('Set2', n_colors=len(seg_df)),
    startangle=90, textprops={'fontsize': 8}
)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')
ax.set_title('Distribution of Total Absolute Error\n(Where does error concentrate?)')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_smape_by_segment.png'), dpi=150, bbox_inches='tight')
plt.show()

### Teaching: SMAPE vs WMAE Rankings

Notice the contrast:
- **SMAPE** is highest for **low-volume** items (because small absolute errors become large percentage errors when the denominator is small)
- **Error Share** is concentrated in **high-volume** items (because their absolute errors are larger in total)

If you optimize for SMAPE, you would focus improvement efforts on low-volume items where the economic impact is minimal. **WMAE** (Weighted MAE) corrects this by weighting each error proportionally to actual demand -- making it a much more appropriate metric for retail inventory decisions.

**Recommendation**: Use WMAE as your primary evaluation metric for retail demand forecasting. SMAPE is fine for academic benchmarks but can be actively misleading for business decisions.

In [ ]:
# --- Direct comparison: SMAPE ranking vs WMAE ranking ---
rank_df = seg_df[['Segment', 'SMAPE', 'WMAE']].copy()
rank_df['SMAPE_Rank'] = rank_df['SMAPE'].rank(ascending=False).astype(int)
rank_df['WMAE_Rank']  = rank_df['WMAE'].rank(ascending=False).astype(int)
rank_df['Rank_Diff']  = rank_df['SMAPE_Rank'] - rank_df['WMAE_Rank']

print("SMAPE Ranking vs WMAE Ranking by Segment")
print("(Rank 1 = worst performance)")
print("="*70)
display(rank_df.round(4))
print("\nNotice: SMAPE says low-volume items are 'worst', but WMAE correctly")
print("identifies high-volume items as where errors matter most economically.")

---

## Section 4: Additional Segmentations

Beyond volume, we segment errors by:
- **Stockout rate**: How does censoring affect forecast quality?
- **Day of week**: Are weekends harder to predict?
- **Product category**: Which categories are most/least predictable?

### 4.1 By Stockout Rate

In [ ]:
# --- Segment SPs into stockout groups ---
eval_sp_sor = np.array([sp_stockout_rate.get(sp, 0) for sp in sp_eval])

def assign_stockout_group(sor):
    if sor < 0.1:
        return 'Low (< 10%)'
    elif sor < 0.3:
        return 'Medium (10-30%)'
    else:
        return 'High (> 30%)'

so_groups = np.array([assign_stockout_group(s) for s in eval_sp_sor])
so_group_order = ['Low (< 10%)', 'Medium (10-30%)', 'High (> 30%)']

so_results = []
for grp in so_group_order:
    mask = so_groups == grp
    if mask.sum() == 0:
        continue
    m = compute_metrics(y_eval[mask], p_ens[mask])
    so_results.append({'Stockout Group': grp, 'Count': int(mask.sum()), **m})

so_df = pd.DataFrame(so_results)
print("METRICS BY STOCKOUT RATE GROUP")
print("=" * 80)
display(so_df.round(4))

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(so_df))
w = 0.3
ax.bar(x - w/2, so_df['MAE'], w, label='MAE', color='steelblue')
ax.bar(x + w/2, so_df['WMAE'], w, label='WMAE', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(so_df['Stockout Group'])
ax.set_ylabel('Error')
ax.set_title('MAE and WMAE by Stockout Rate Group')
ax.legend()
for i, (m, wm) in enumerate(zip(so_df['MAE'], so_df['WMAE'])):
    ax.text(i - w/2, m + 0.005, f'{m:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, wm + 0.005, f'{wm:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_error_by_stockout_rate.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.2 By Day of Week

In [ ]:
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_results = []

for d in range(7):
    mask = dow_eval == d
    if mask.sum() == 0:
        continue
    m = compute_metrics(y_eval[mask], p_ens[mask])
    dow_results.append({'Day': dow_names[d], 'Count': int(mask.sum()),
                        'MAE': m['MAE'], 'RMSE': m['RMSE'], 'Bias': m['Bias']})

dow_df = pd.DataFrame(dow_results)
print("METRICS BY DAY OF WEEK")
print("=" * 60)
display(dow_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['coral' if d in ['Sat', 'Sun'] else 'steelblue' for d in dow_df['Day']]
bars = ax.bar(dow_df['Day'], dow_df['MAE'], color=colors, edgecolor='gray', lw=0.5)
ax.set_ylabel('MAE')
ax.set_title('MAE by Day of Week (weekends in coral)')
for bar, val in zip(bars, dow_df['MAE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_mae_by_dow.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.3 By Product Category (Top 5)

In [ ]:
# Find top 5 first_category_ids by frequency
cat_counts = pd.Series(cat1_eval).value_counts()
top5_cats = cat_counts.head(5).index.tolist()

cat_results = []
for cat in top5_cats:
    mask = cat1_eval == cat
    if mask.sum() == 0:
        continue
    m = compute_metrics(y_eval[mask], p_ens[mask])
    cat_results.append({'Category': cat, 'Count': int(mask.sum()),
                        'MAE': m['MAE'], 'RMSE': m['RMSE'],
                        'SMAPE': m['SMAPE'], 'WMAE': m['WMAE'], 'Bias': m['Bias']})

cat_df = pd.DataFrame(cat_results)
print("METRICS BY TOP 5 PRODUCT CATEGORIES (first_category_id)")
print("=" * 80)
display(cat_df.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(cat_df))
w = 0.3
ax.bar(x - w/2, cat_df['MAE'], w, label='MAE', color='steelblue')
ax.bar(x + w/2, cat_df['WMAE'], w, label='WMAE', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'Cat {c}' for c in cat_df['Category']])
ax.set_ylabel('Error')
ax.set_title('MAE and WMAE by Product Category (Top 5)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_error_by_category.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## Section 5: Prediction Interval Calibration

### Why Calibration Matters

Our Q10 and Q90 models define an 80% prediction interval. If the model is well-calibrated, exactly 80% of actual observations should fall within this band.

**Under-coverage** (actual coverage < 80%) means the model **underestimates uncertainty**. In an inventory context, this is dangerous: if prediction intervals are too narrow, safety stock calculations based on these intervals will lead to **higher stockout rates** than expected.

**Over-coverage** (actual coverage > 80%) means the model is conservative -- less risky for inventory but potentially leads to excess stock.

In [ ]:
# --- Overall Coverage ---
in_interval = (y_eval >= preds['q10']) & (y_eval <= preds['q90'])
overall_coverage = in_interval.mean()

print("=" * 60)
print("PREDICTION INTERVAL CALIBRATION")
print("=" * 60)
print(f"Expected coverage (Q10 to Q90): 80.0%")
print(f"Actual coverage (overall):      {overall_coverage*100:.1f}%")
gap = (overall_coverage - 0.80) * 100
if gap < -2:
    print(f"Status: UNDER-COVERED by {abs(gap):.1f}pp -- model underestimates uncertainty")
elif gap > 2:
    print(f"Status: OVER-COVERED by {gap:.1f}pp -- model is conservative")
else:
    print(f"Status: WELL-CALIBRATED (within 2pp of target)")

In [ ]:
# --- Coverage by Volume Segment ---
cov_results = []
for seg in segment_order:
    mask = segment_labels == seg
    if mask.sum() == 0:
        continue
    cov = in_interval[mask].mean()
    avg_width = np.mean(preds['q90'][mask] - preds['q10'][mask])
    cov_results.append({
        'Segment': seg,
        'Count': int(mask.sum()),
        'Coverage (%)': cov * 100,
        'Expected (%)': 80.0,
        'Gap (pp)': (cov - 0.80) * 100,
        'Avg PI Width': avg_width,
    })

cov_df = pd.DataFrame(cov_results)
print("\n80% PI COVERAGE BY VOLUME SEGMENT")
print("=" * 80)
display(cov_df.round(2))

# Visualization
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d9534f' if g < -3 else '#5cb85c' if abs(g) <= 3 else '#f0ad4e'
          for g in cov_df['Gap (pp)']]
bars = ax.barh(cov_df['Segment'], cov_df['Coverage (%)'], color=colors, edgecolor='gray', lw=0.5)
ax.axvline(80, color='black', ls='--', lw=1.5, label='Target (80%)')
ax.set_xlabel('Coverage (%)')
ax.set_title('80% PI Coverage by Volume Segment\n(red = under-covered, green = well-calibrated, orange = over-covered)')
ax.legend(fontsize=9)
for bar, val, gap in zip(bars, cov_df['Coverage (%)'], cov_df['Gap (pp)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}% ({gap:+.1f}pp)', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb04_pi_coverage_by_segment.png'), dpi=150, bbox_inches='tight')
plt.show()
cov_df.to_csv(os.path.join(NB_OUT, 'nb04_pi_coverage_by_segment.csv'), index=False)
print(f'Saved nb04_pi_coverage_by_segment.csv to {NB_OUT}')

### Teaching: Calibration Implications for Inventory

If your 80% prediction interval only covers 70% of actual values, then:
- Safety stock calculated from Q90 will be **insufficient** 30% of the time (instead of the expected 10%)
- This translates directly into **higher stockout rates** and **lower fill rates**
- You would need to use a higher quantile (e.g., Q95) to achieve the intended service level

Always check PI calibration before using quantile forecasts for inventory optimization.

---

## Section 6: Diagnostic Time Series Plots

Aggregate metrics tell us the average story. To truly understand model behavior, we need to look at individual store-products across the forecast horizon.

We select 6 examples: 2 low-volume, 2 medium-volume, and 2 high-volume store-products.

In [ ]:
# --- Select 6 representative SPs ---
unique_sps = np.unique(sp_eval)
sp_avg_arr = np.array([sp_avg_demand.get(sp, 0) for sp in unique_sps])

# Low volume: avg demand < 0.5
low_sps = unique_sps[sp_avg_arr < 0.5]
# Medium volume: 1.0 - 2.0
med_sps = unique_sps[(sp_avg_arr >= 1.0) & (sp_avg_arr < 2.0)]
# High volume: > 5.0
high_sps = unique_sps[sp_avg_arr >= 5.0]

np.random.seed(42)
selected = []
for pool, label in [(low_sps, 'Low'), (med_sps, 'Medium'), (high_sps, 'High')]:
    if len(pool) >= 2:
        chosen = np.random.choice(pool, 2, replace=False)
    elif len(pool) == 1:
        chosen = pool[:1]
    else:
        chosen = np.random.choice(unique_sps, 2, replace=False)
    for sp in chosen:
        selected.append((sp, label))

print(f"Selected {len(selected)} store-products for diagnostic plots:")
for sp, label in selected:
    avg = sp_avg_demand.get(sp, 0)
    print(f"  SP={sp}, Volume={label}, Avg Demand={avg:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Diagnostic Time Series: Actual vs Forecast with 80% PI\n'
             '(7-Day Eval Period)', fontsize=13, fontweight='bold')

for idx, (ax, (sp, vol_label)) in enumerate(zip(axes.flat, selected)):
    mask = sp_eval == sp
    if mask.sum() == 0:
        ax.set_title(f'SP={sp} -- No data')
        continue

    y_sp  = y_eval[mask]
    p_sp  = p_ens[mask]
    q10_sp = preds['q10'][mask]
    q90_sp = preds['q90'][mask]
    days = np.arange(len(y_sp))

    # PI band
    ax.fill_between(days, q10_sp, q90_sp, alpha=0.25, color='steelblue', label='80% PI')
    # Actual
    ax.plot(days, y_sp, 'ko-', ms=6, lw=1.5, label='Actual', zorder=5)
    # Forecast
    ax.plot(days, p_sp, 'rs-', ms=5, lw=1.5, alpha=0.8, label='Forecast', zorder=4)

    mae_sp = mean_absolute_error(y_sp, p_sp)
    cov_sp = np.mean((y_sp >= q10_sp) & (y_sp <= q90_sp)) * 100
    avg_dem = sp_avg_demand.get(sp, 0)

    ax.set_title(f'{vol_label} Volume | SP={sp}\n'
                 f'MAE={mae_sp:.3f}, Cov={cov_sp:.0f}%, Avg={avg_dem:.2f}',
                 fontsize=9)
    ax.set_xlabel('Day')
    ax.set_ylabel('Demand')
    ax.set_xticks(days)
    ax.set_xticklabels([f'D{d+1}' for d in days])
    if idx == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(os.path.join(NB_OUT, 'nb04_diagnostic_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()

**Observations from the diagnostic plots:**

- **Low-volume items**: Demand is often zero or near-zero. The forecast may predict small positive values, leading to high SMAPE but trivial MAE. PI bands tend to be narrow.
- **Medium-volume items**: More regular patterns. The forecast generally tracks well, and PIs provide meaningful coverage.
- **High-volume items**: Larger absolute errors but forecast captures the overall trend. PI width is proportional to demand variability.

These visual diagnostics complement aggregate metrics -- always inspect individual time series to understand *how* your model fails.

---

## Section 7: Key Takeaways

### 1. SMAPE is Misleading for Heterogeneous Demand

SMAPE treats all observations equally, giving disproportionate weight to low-volume items where small absolute errors become large percentage errors. For retail demand forecasting with products ranging from 0.01 to 50+ kg/day, SMAPE provides a distorted view of model performance.

### 2. Use WMAE or Weighted Metrics for Retail

**WMAE** (Weighted Mean Absolute Error, where weights = actual demand) focuses attention on errors that matter economically. A 1 kg error on a product selling 10 kg/day has 10x the weight of a 1 kg error on a product selling 1 kg/day. This directly aligns with business impact: high-volume errors cost more in lost revenue and waste.

### 3. Check Calibration of Prediction Intervals

An 80% PI should cover 80% of actual values. Under-coverage means:
- Safety stock will be insufficient
- Actual stockout rates will exceed planned levels
- Service level targets will be missed

Always verify PI calibration, especially across demand segments -- coverage can vary significantly between low and high-volume products.

### 4. Segment Analysis Reveals Where to Focus Improvement Efforts

Aggregate metrics hide important variation. By segmenting errors by volume, stockout rate, day-of-week, and category, you can identify:
- Which segments drive most total error (focus here for biggest impact)
- Which segments have the worst bias (systematic improvement opportunities)
- Which segments have poor PI coverage (uncertainty model improvements needed)

### Summary Decision Framework

| If you want to... | Use this metric |
|---|---|
| Minimize total inventory cost | WMAE (weights by economic value) |
| Benchmark across datasets | SMAPE (scale-free but biased toward low-volume) |
| Detect systematic bias | Bias = mean(pred - actual) |
| Set safety stock levels | PI Coverage by segment |
| Compare model families | MAE + RMSE + Correlation together |

In [ ]:
# --- Final Summary ---
print("=" * 80)
print("NOTEBOOK SUMMARY")
print("=" * 80)
print(f"\nStore-products analyzed: {len(np.unique(sp_eval)):,}")
print(f"Eval observations:      {len(y_eval):,}")
print(f"\nBest model (by MAE): {metrics_df['MAE'].idxmin()} "
      f"(MAE = {metrics_df['MAE'].min():.4f})")
print(f"Best model (by WMAE): {metrics_df['WMAE'].idxmin()} "
      f"(WMAE = {metrics_df['WMAE'].min():.4f})")
print(f"\n80% PI Coverage: {overall_coverage*100:.1f}%")
print(f"\nVolume segment with highest error share: "
      f"{seg_df.loc[seg_df['Error Share %'].idxmax(), 'Segment']} "
      f"({seg_df['Error Share %'].max():.1f}%)")
print(f"\nVolume segment with highest SMAPE: "
      f"{seg_df.loc[seg_df['SMAPE'].idxmax(), 'Segment']} "
      f"({seg_df['SMAPE'].max():.1f}%)")
print("\n" + "=" * 80)
print("END OF NOTEBOOK")
print("=" * 80)